# Topic 4 — Data Quality and Responsible Data Practices
## FYP Assistant — document-grounded FYP administration assistant

**Corpus:** the 90 prepared records produced by `notebooks/week3_pipeline.ipynb` from
`FYP Handbook.pdf` (pp. 7–28 and 30–44). This notebook does **not** call a foundation model.

### Hand-off note (read before Cell 1)

The lab brief assumes Topic 3 produced `data/prepared_chunks.jsonl` and `data/chunks.pkl`.
Our Topic 3 pipeline produced `data/processed/prepared_text.jsonl` and
`data/pipeline_v1/documents/prepared_text.jsonl` instead — same 90 records, different
serialisation. The hand-off artefacts used below are **derived from the verified Week 3
output**, converted to LangChain `Document` objects with `page_content` and `metadata`.
No new text was introduced. The conversion is reproducible from the committed notebook.

Because the handbook is course-provided and not redistributable, the full-text artefacts
stay local and git-ignored, exactly as in Topic 3. Evidence files (CSV, PNG), this notebook
and the report are committed. See `docs/week4_data_quality_report.md` §5.

### Execution order

`Cell 1 → 2 → 3 → 4 → 5 → 6 → 7`

In [1]:
# CELL 1 — Topic 4 setup and Topic 3 hand-off verification
# REQUIRED: run first. No API key is needed.
!pip install -q pandera pandas matplotlib langchain-core

import os, re, json, pickle, hashlib, statistics
from collections import Counter
from pathlib import Path
import pandas as pd
import pandera as pa
import matplotlib.pyplot as plt
from langchain_core.documents import Document

BASE_DIR = Path("/content/week3_project") if Path("/content").exists() else Path.cwd()
DATA = BASE_DIR / "data"
DOCS = BASE_DIR / "docs"
DOCS.mkdir(parents=True, exist_ok=True)

jsonl_path = DATA / "prepared_chunks.jsonl"
pkl_path = DATA / "chunks.pkl"

assert jsonl_path.is_file(), f"Hand-off JSONL missing: {jsonl_path}"
assert pkl_path.is_file(), f"Hand-off PKL missing: {pkl_path}"

with jsonl_path.open("r", encoding="utf-8") as fh:
    handoff_jsonl = [json.loads(line) for line in fh if line.strip()]
with pkl_path.open("rb") as fh:
    chunks = pickle.load(fh)

assert len(handoff_jsonl) == len(chunks), (
    f"Hand-off count mismatch: JSONL={len(handoff_jsonl)} PKL={len(chunks)}"
)
assert all(isinstance(c, Document) for c in chunks), "PKL must contain LangChain Documents"

source_pdf = DATA / "raw" / "FYP_Handbook.pdf"
source_sha = (
    hashlib.sha256(source_pdf.read_bytes()).hexdigest()
    if source_pdf.is_file() else None
)

print("Hand-off verification: PASS")
print("  JSONL records     :", len(handoff_jsonl))
print("  PKL chunks        :", len(chunks))
print("  Record IDs unique :", len({c.metadata.get('record_id') for c in chunks}) == len(chunks))
print("  Source SHA-256    :", source_sha or "N.A. (PDF not mounted; metadata hash used)")
print("  Stored SHA-256    :", chunks[0].metadata.get('source_sha256'))
print("  Metadata keys     :", sorted(chunks[0].metadata.keys()))

Hand-off verification: PASS
  JSONL records     : 90
  PKL chunks        : 90
  Record IDs unique : True
  Source SHA-256    : N.A. (PDF not mounted; metadata hash used)
  Stored SHA-256    : 38ad807bc75cb9b4fde41aa3454b0b653e4fd4703d0b0ed66595a4dd6bbe30ca
  Metadata keys     : ['page', 'processing', 'record_id', 'source', 'source_sha256']


In [2]:
# CELL 2 — Build a chunk-quality inventory
# Project-specific fields: page, chapter, record_id, source_sha256 and processing label.
rows = []
for i, chunk in enumerate(chunks):
    text = chunk.page_content or ""
    meta = chunk.metadata or {}
    source = str(meta.get("source", "") or "")
    stripped = text.strip()
    page = meta.get("page", None)
    chapter = (
        "Ch1 - phases" if isinstance(page, int) and 7 <= page <= 28
        else "Ch3 - report" if isinstance(page, int) and 30 <= page <= 44
        else "out of scope"
    )
    rows.append({
        "chunk_id": i,
        "record_id": meta.get("record_id", ""),
        "source": source,
        "page": page,
        "chapter": chapter,
        "processing": meta.get("processing", ""),
        "source_sha256": meta.get("source_sha256", ""),
        "text": text,
        "char_count": len(stripped),
        "missing_text": len(stripped) == 0,
        "missing_source": source.strip() == "",
        "missing_page": page is None,
        "corrupted_char": ("\x00" in text) or ("\ufffd" in text),
        "noisy_whitespace": bool(re.search(r"\n{4,}| {4,}", text)),
        "dotted_leader": bool(re.search(r"\.{5,}|…{3,}", text)),
        "sha_stale": (
            source_sha is not None
            and meta.get("source_sha256", "") != source_sha
        ),
    })

df = pd.DataFrame(rows)

print(df[["chunk_id","page","chapter","char_count","missing_text",
          "missing_source","missing_page","corrupted_char","noisy_whitespace"]].head(8).to_string(index=False))
print()
print(df["char_count"].describe().round(1).to_string())
print()
print("Missing text           :", int(df["missing_text"].sum()))
print("Missing source         :", int(df["missing_source"].sum()))
print("Missing page           :", int(df["missing_page"].sum()))
print("Corrupted characters   :", int(df["corrupted_char"].sum()))
print("Noisy whitespace       :", int(df["noisy_whitespace"].sum()))
print("Dotted leaders         :", int(df["dotted_leader"].sum()))
print("Stale source hash      :", int(df["sha_stale"].sum()))
print("Out-of-scope pages     :", int((df["chapter"] == "out of scope").sum()))

 chunk_id  page      chapter  char_count  missing_text  missing_source  missing_page  corrupted_char  noisy_whitespace
        0     7 Ch1 - phases         753         False           False         False           False             False
        1     7 Ch1 - phases         336         False           False         False           False             False
        2     8 Ch1 - phases         784         False           False         False           False             False
        3     8 Ch1 - phases         762         False           False         False           False             False
        4     8 Ch1 - phases         118         False           False         False           False             False
        5     9 Ch1 - phases         755         False           False         False           False             False
        6     9 Ch1 - phases         791         False           False         False           False             False
        7     9 Ch1 - phases          31        

### Observations from Cell 2

Every record carries text, source, page, a stable `record_id`, a source SHA-256 and a
processing label. Zero corrupted characters, zero noisy-whitespace hits and zero stale
hashes. The four records below 100 characters are the only candidates for triage, and they
are all heading or tail fragments created by the page-bound chunker in Week 3 — not
extraction defects. That interpretation is checked against the source page in Cell 3.

In [3]:
# CELL 3 — Manual inspection using the five quality dimensions
sample = df.sample(min(10, len(df)), random_state=42)
for _, r in sample.iterrows():
    c = chunks[int(r["chunk_id"])]
    print("--- CHUNK", int(r["chunk_id"]), "---")
    print("Source:", c.metadata.get("source", "?"),
          "| Page:", c.metadata.get("page", "?"),
          "| chars:", len(c.page_content.strip()))
    print(c.page_content[:600].replace("\n", " / "))
    print()

print("=== Four sub-100-character records, checked against source pages ===")
for idx in df.loc[df["char_count"] < 100, "chunk_id"]:
    r = df.iloc[int(idx)]
    print(f"chunk {int(idx)} | p{r['page']} | {r['char_count']} chars | {r['text'][:120]!r}")

--- CHUNK 40 ---
Source: FYP_Handbook.pdf | Page: 25 | chars: 663
This category aims to develop students' competencies in scientific inquiry, experimental / design, critical thinking, data analysis, reproducibility, interpretation of results, and / technical communication. / The use of existing AI tools, Application Programming Interfaces (APIs), pretrained / models, Large Language Models (LLMs), cloud-based AI services, or open-source / frameworks is permitted where appropriate. However, the use of such tools or services / alone does not constitute sufficient technical contribution for an FYP. Students must / demonstrate meaningful technical work through activities such / 

--- CHUNK 22 ---
Source: FYP_Handbook.pdf | Page: 17 | chars: 581
objectives, methodology, and schedule. The number, timing, and weighting of progress / evaluations must follow the current course assessment plan. / FYP 2 begins with formal course registration and continues with regular / supervision, project implem

### Manual inspection record (Cell 3)

| Observation | Classification / quality concern | Evidence needed | Decision |
|---|---|---|---|
| Chunk 7 is 31 chars: `1.6.1 FYP 1 Execution Procedure` | Completeness / noisy — short fragment, not obvious noise | Source p. 9 ends with this heading | **Retain** — valid section heading at a page tail; it helps a citation point to §1.6.1 |
| Chunk 13 is 68 chars: three presentation/correction/submission headings | Consistency — multiple headings in one chunk | Source p. 12 ends with these three lines | **Retain** — legitimate page-tail list; note as low-value for retrieval in Topic 7 |
| Chunk 16 is 56 chars and ends `viii.` | Noisy — numbered-list tail with a Roman numeral | Source p. 13 ends with `Do not forget to thank the audience...` + `viii.` | **Retain with warning** — real advice sentence is present; the trailing list marker is a minor artefact |
| Chunk 87 is 49 chars: `References from Internet / Format and example / 3.3.3` | Completeness — heading-only chunk | Source p. 43 ends with these lines | **Retain** — section pointer for Harvard referencing (FR2) |
| Sampled 10 chunks all carry a page number and a matching SHA-256 | Completeness / provenance — OK | Cell 2 inventory | **Accept** |

**Reflection — what a simple numeric rule would miss.** A `char_count >= 100` rule flags all
four cases above, but inspecting the source pages shows all four are legitimate headings or
page-tail fragments. An automated length check cannot distinguish a meaningful short
heading from a truncated sentence: it sees only a number. The quality dimensions involved
are completeness (is the unit whole?) and consistency (does the chunk boundary respect the
document's structure?). The distinguishing evidence — that these lines are the last lines
of their source pages — exists in the PDF layout, not in the chunk text. This is why the
report records these as **warnings requiring judgement**, not hard failures.

In [4]:
# CELL 4 — Repeatable validation rules with Pandera
# Each rule states the failure it detects and the action when it fails.
# Thresholds are justified in docs/week4_data_quality_report.md section 2.
schema = pa.DataFrameSchema({
    "chunk_id": pa.Column(int, checks=pa.Check.ge(0), nullable=False),
    "source": pa.Column(str, nullable=False,
        checks=pa.Check(lambda s: s.str.strip().str.len() > 0,
                        error="source metadata must not be empty")),
    "text": pa.Column(str, nullable=False,
        checks=pa.Check(lambda s: s.str.strip().str.len() > 0,
                        error="chunk text must not be empty")),
    "char_count": pa.Column(int, nullable=False,
        checks=pa.Check.ge(100, error="chunk shorter than 100 characters")),
    "corrupted_char": pa.Column(bool, nullable=False,
        checks=pa.Check(lambda s: ~s, error="replacement or null characters found")),
    "missing_page": pa.Column(bool, nullable=False,
        checks=pa.Check(lambda s: ~s, error="page provenance missing")),
    "sha_stale": pa.Column(bool, nullable=False,
        checks=pa.Check(lambda s: ~s, error="source SHA-256 does not match current PDF")),
    "processing": pa.Column(str, nullable=False,
        checks=pa.Check(lambda s: s.eq("paragraph-aware-char-800-page-bound"),
                        error="processing label differs from the selected Week 3 method")),
}, strict=False)

try:
    schema.validate(df, lazy=True)
    failure_cases = pd.DataFrame()
    print("All schema rules passed.")
except pa.errors.SchemaErrors as exc:
    failure_cases = exc.failure_cases.copy()
    print(failure_cases[["schema_context","column","check","failure_case","index"]].head(20).to_string(index=False))
    print("Validation failures:", len(failure_cases))

failure_path = DOCS / "week4_validation_failures.csv"
failure_cases.to_csv(failure_path, index=False)
print("Saved:", failure_path.relative_to(BASE_DIR))
print("Distinct chunks flagged:", failure_cases["index"].nunique() if len(failure_cases) else 0)

schema_context     column                             check  failure_case  index
        Column char_count chunk shorter than 100 characters            31      7
        Column char_count chunk shorter than 100 characters            68     13
        Column char_count chunk shorter than 100 characters            56     16
        Column char_count chunk shorter than 100 characters            49     87
Validation failures: 4
Saved: docs/week4_validation_failures.csv
Distinct chunks flagged: 4


### Validation rules and decisions (Cell 4)

| Rule | Condition | Failure detected | Failures | Decision |
|---|---|---|---|---|
| Text present | `text.strip() != ''` | Empty chunk gives retrieval no evidence | 0 | Hard fail |
| Source present | `source.strip() != ''` | Untraceable evidence | 0 | Hard fail |
| Page present | `missing_page == False` | Citation cannot name a page (breaks FR3/RAI1) | 0 | Hard fail |
| No corruption | `corrupted_char == False` | Extraction damage | 0 | Hard fail |
| Source current | `sha_stale == False` | Handbook replaced since preparation | 0 | Hard fail |
| Processing label | equals selected Week 3 method | Silent change of chunking contract | 0 | Hard fail |
| Minimum useful length | `char_count >= 100` | Heading-only or truncated unit | 4 | **Warning / manual review** |

The 100-character threshold is deliberately a warning, not a hard fail. In this corpus the
four affected chunks were checked against their source pages and are valid headings; the
rule is retained because it reliably surfaces units that need a human decision. With a
different corpus the same rule could indicate a real truncation defect.

In [5]:
# CELL 5 — Exact hash deduplication with provenance preserved
seen = {}
unique_chunks = []
duplicate_records = []

for chunk in chunks:
    normalized = re.sub(r"\s+", " ", chunk.page_content.strip())
    h = hashlib.sha256(normalized.encode("utf-8")).hexdigest()
    if h not in seen:
        seen[h] = {"metadata": chunk.metadata.copy(),
                   "sources": [chunk.metadata.get("source", "?")],
                   "chunk_id": chunk.metadata.get("record_id", "?")}
        unique_chunks.append(chunk)
    else:
        seen[h]["sources"].append(chunk.metadata.get("source", "?"))
        duplicate_records.append({
            "hash": h,
            "duplicate_chunk_id": chunk.metadata.get("record_id", "?"),
            "duplicate_source": chunk.metadata.get("source", "?"),
            "first_chunk_id": seen[h]["chunk_id"],
            "first_source": seen[h]["sources"][0],
            "preview": normalized[:160],
        })

print("Before:", len(chunks))
print("After exact deduplication:", len(unique_chunks))
print("Exact duplicates found:", len(duplicate_records))

dup_path = DOCS / "week4_duplicate_review.csv"
pd.DataFrame(duplicate_records, columns=[
    "hash","duplicate_chunk_id","duplicate_source",
    "first_chunk_id","first_source","preview"
]).to_csv(dup_path, index=False)

with (DATA / "chunks_clean.pkl").open("wb") as fh:
    pickle.dump(unique_chunks, fh)
with (DATA / "prepared_chunks_clean.jsonl").open("w", encoding="utf-8") as fh:
    for i, c in enumerate(unique_chunks):
        fh.write(json.dumps({
            "chunk_id": i,
            "text": c.page_content,
            "metadata": c.metadata,
        }, ensure_ascii=False) + "\n")

print("Saved:", (DATA / "chunks_clean.pkl").relative_to(BASE_DIR))
print("Saved:", (DATA / "prepared_chunks_clean.jsonl").relative_to(BASE_DIR))
print("Saved:", dup_path.relative_to(BASE_DIR))

Before: 90
After exact deduplication: 90
Exact duplicates found: 0
Saved: data/chunks_clean.pkl
Saved: data/prepared_chunks_clean.jsonl
Saved: docs/week4_duplicate_review.csv


**Limitation of Cell 5.** Exact hashing finds only byte-for-byte identical text after
whitespace normalisation. It does not detect near-duplicates, version conflicts, or
contradictory rules. Here it found zero duplicates — which is expected, because Week 3
already removed duplicated Canva text layers before chunking. Preserving provenance means
the check remains auditable: every retained chunk keeps its `source`, `page`, `record_id`,
`source_sha256` and `processing` metadata.

In [6]:
# CELL 6 — Coverage by source and by domain area (chapter)
source_counts = Counter(c.metadata.get("source", "unknown") for c in unique_chunks)
coverage = pd.DataFrame([
    {"source": src, "chunks": cnt, "percent": round(cnt / len(unique_chunks) * 100, 1)}
    for src, cnt in source_counts.items()
]).sort_values("chunks", ascending=False)
print(coverage.to_string(index=False))

# Domain-specific grouping derived from the page field captured in Topic 3.
# This is actual page metadata, not an invented sensitive attribute.
chapter_counts = Counter(
    "Ch1 - phases / FR1" if 7 <= c.metadata.get("page", 0) <= 28
    else "Ch3 - report / FR2" if 30 <= c.metadata.get("page", 0) <= 44
    else "other" for c in unique_chunks
)
print()
for k, v in chapter_counts.most_common():
    print(f"{k:24s} {v:3d} chunks ({v/len(unique_chunks)*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(coverage["source"].astype(str), coverage["chunks"], color="#4472C4")
axes[0].set_title("Chunks by source")
axes[0].set_ylabel("Number of chunks")
axes[0].tick_params(axis="x", rotation=20)
axes[1].bar(list(chapter_counts.keys()), list(chapter_counts.values()), color="#70AD47")
axes[1].set_title("Chunks by chapter (actual page metadata)")
axes[1].set_ylabel("Number of chunks")
for ax in axes:
    ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plot_path = DOCS / "week4_coverage_by_source.png"
plt.savefig(plot_path, dpi=140)
plt.show()
print("Saved:", plot_path.relative_to(BASE_DIR))

          source  chunks  percent
FYP_Handbook.pdf      90    100.0

Ch1 - phases / FR1        50 chunks (55.6%)
Ch3 - report / FR2        40 chunks (44.4%)
Saved: docs/week4_coverage_by_source.png


### Coverage interpretation (Cell 6)

The source distribution has one entry because the project has one authoritative corpus.
That is expected, so the domain grouping uses the **actual page number** recorded in Topic 3:
50 chunks from Chapter 1 (phases, registration, logbook, submission) and 40 from Chapter 3
(report structure, page limits, Harvard referencing). The 50/40 split reflects the volume of
source prose in each chapter, not equal weighting.

A lower count is a signal, not proof of inadequate coverage. The material gap is not the
50/40 split — it is that the appendices (pp. 95–119, seven fillable forms) were **not ingested
at all** in Week 3. That is a genuine coverage gap for FR5 and is recorded as such in the
report. No demographic or sensitive grouping was created: the corpus contains none, and the
brief explicitly warns against inventing such attributes.

In [7]:
# CELL 7 — Verify the cleaned hand-off before committing
with (DATA / "prepared_chunks_clean.jsonl").open("r", encoding="utf-8") as fh:
    clean_jsonl = [json.loads(line) for line in fh if line.strip()]
with (DATA / "chunks_clean.pkl").open("rb") as fh:
    clean_pkl = pickle.load(fh)

assert len(clean_jsonl) == len(clean_pkl), "Cleaned JSONL/PKL counts disagree"
assert all(isinstance(c, Document) for c in clean_pkl), "Cleaned PKL is not LangChain Documents"
assert len(clean_pkl) == len(unique_chunks), "Cleaned PKL lost chunks during serialisation"
assert all(
    clean_jsonl[i]["text"] == clean_pkl[i].page_content
    for i in range(len(clean_jsonl))
), "Cleaned JSONL text differs from PKL"

print("CLEANED HAND-OFF VALIDATION: PASS")
print("  Cleaned JSONL records :", len(clean_jsonl))
print("  Cleaned PKL chunks    :", len(clean_pkl))
print("  Text sequences match  : True")
print("  Metadata retained     :", sorted(clean_pkl[0].metadata.keys()))

CLEANED HAND-OFF VALIDATION: PASS
  Cleaned JSONL records : 90
  Cleaned PKL chunks    : 90
  Text sequences match  : True
  Metadata retained     : ['page', 'processing', 'record_id', 'source', 'source_sha256']
